## Export PR - CHIRPS

In [3]:
# export_data.py
# Export monthly CHIRPS data (1994–2025) for study areas in the Amazon/Pantanal.
# Generates an Excel file with monthly precipitation per pixel (Latitude x Longitude).

import time

import ee
import numpy as np
import pandas as pd


# =============================================================================
# GOOGLE EARTH ENGINE AUTHENTICATION AND INITIALIZATION
# =============================================================================

def initialize_gee(project: str = 'local-abbey-453514-e5') -> None:
    """Authenticate and initialize Google Earth Engine."""
    try:
        ee.Authenticate()
        ee.Initialize(project=project)
        print("✅ GEE authenticated and initialized!")
    except ee.EEException as e:
        print(f"❌ Authentication error: {e}")
        raise


# =============================================================================
# HELPER FUNCTIONS
# =============================================================================

def create_reduce_region_function(geometry,
                                  reducer=ee.Reducer.mean(),
                                  scale=5566,
                                  crs='EPSG:4326',
                                  bestEffort=True,
                                  maxPixels=1e13,
                                  tileScale=4):
    """Return a function that reduces an image to statistics over a geometry."""

    def reduce_region_function(img):
        stat = img.reduceRegion(
            reducer=reducer,
            geometry=geometry,
            scale=scale,
            crs=crs,
            bestEffort=bestEffort,
            maxPixels=maxPixels,
            tileScale=tileScale,
        )
        return ee.Feature(geometry, stat).set({'millis': img.date().millis()})

    return reduce_region_function


def fc_to_dict(fc):
    """Convert a FeatureCollection into a Python dictionary."""
    prop_names = fc.first().propertyNames()
    prop_lists = fc.reduceColumns(
        reducer=ee.Reducer.toList().repeat(prop_names.size()),
        selectors=prop_names,
    ).get('list')
    return ee.Dictionary.fromLists(prop_names, prop_lists)


def add_date_info(df: pd.DataFrame) -> pd.DataFrame:
    """Add date columns (Timestamp, Year, Month, Day, DOY) to a DataFrame."""
    df['Timestamp'] = pd.to_datetime(df['millis'], unit='ms')
    df['Year'] = pd.DatetimeIndex(df['Timestamp']).year
    df['Month'] = pd.DatetimeIndex(df['Timestamp']).month
    df['Day'] = pd.DatetimeIndex(df['Timestamp']).day
    df['DOY'] = pd.DatetimeIndex(df['Timestamp']).dayofyear
    return df


def aggregate_to_monthly(daily_collection: ee.ImageCollection) -> ee.ImageCollection:
    """Aggregate a daily collection into a monthly collection (sum)."""
    start_date = ee.Date(daily_collection.first().get('system:time_start'))
    end_date = ee.Date(
        daily_collection.sort('system:time_start', False).first().get('system:time_start')
    )

    n_months = end_date.difference(start_date, 'month').round()
    months = ee.List.sequence(0, n_months.subtract(1))

    def create_monthly_image(i):
        start = start_date.advance(i, 'month')
        end = start.advance(1, 'month')
        monthly_data = daily_collection.filterDate(start, end).sum()
        return monthly_data.rename('precipitation').set({
            'system:time_start': start.millis(),
            'system:index': start.format('YYYY_MM'),
        })

    return ee.ImageCollection.fromImages(months.map(create_monthly_image))


def extract_lat_lon_pixel(image: ee.Image, geometry: ee.Geometry, bands: list):
    """Extract latitude, longitude and band values for each pixel within the geometry."""
    image_with_coords = image.addBands(ee.Image.pixelLonLat())
    bands_to_select = ['longitude', 'latitude'] + bands

    coords = image_with_coords.select(bands_to_select).reduceRegion(
        reducer=ee.Reducer.toList(),
        geometry=geometry,
        scale=5566,
        bestEffort=True,
    )

    band_values = [
        np.array(ee.List(coords.get(b)).getInfo()).astype(float) for b in bands
    ]
    lat = np.array(ee.List(coords.get('latitude')).getInfo()).astype(float)
    lon = np.array(ee.List(coords.get('longitude')).getInfo()).astype(float)

    return lat, lon, band_values


# =============================================================================
# MAIN CONFIGURATION
# =============================================================================

def main():
    initialize_gee()

    print("=== STARTING CHIRPS 30-YEAR PROCESSING ===")

    # Study areas
    coordinates = '-58.3956807893771, -19.1041167619903, -54.3956912383387, -15.3957931157152'  # Area 1
    # coordinates = '-72.6039770071277, -9.35414223133439, -67.7289897417998, -5.77081825852923'  # Area 2

    x1, y1, x2, y2 = map(float, coordinates.split(","))

    geometry = ee.Geometry.Polygon([[
        [x1, y2], [x2, y2], [x2, y1], [x1, y1], [x1, y2]
    ]])

    start_date, end_date = "1994-01-01", "2025-12-31"

    print(f"Study area: {coordinates}")
    print(f"Period: {start_date} to {end_date}")

    # -------------------------------------------------------------------------
    # 1. Load daily data
    # -------------------------------------------------------------------------
    print("\n1. Loading daily CHIRPS data...")
    t0 = time.time()

    chirps_daily = (
        ee.ImageCollection('UCSB-CHG/CHIRPS/DAILY')
        .filterBounds(geometry)
        .filterDate(start_date, end_date)
        .select('precipitation')
    )

    print(f"   Daily data loaded in {time.time() - t0:.1f}s")

    # -------------------------------------------------------------------------
    # 2. Monthly aggregation
    # -------------------------------------------------------------------------
    print("\n2. Aggregating to monthly data...")
    t0 = time.time()

    chirps_monthly = aggregate_to_monthly(chirps_daily)
    total_images = chirps_monthly.size().getInfo()

    print(f"   Aggregation completed in {time.time() - t0:.1f}s")
    print(f"   Total months processed: {total_images}")

    # -------------------------------------------------------------------------
    # 3. Monthly statistics for the area
    # -------------------------------------------------------------------------
    print("\n3. Computing monthly statistics for the area...")
    t0 = time.time()

    reducer_precip = create_reduce_region_function(
        geometry=geometry, reducer=ee.Reducer.mean(), scale=5566
    )

    precip_stat_fc = ee.FeatureCollection(chirps_monthly.map(reducer_precip))
    precip_dict = fc_to_dict(precip_stat_fc).getInfo()
    precip_df = add_date_info(pd.DataFrame(precip_dict))

    print(f"   Statistics computed in {time.time() - t0:.1f}s")
    print("   First records:")
    display(precip_df.head())

    # -------------------------------------------------------------------------
    # 4. Per-pixel extraction
    # -------------------------------------------------------------------------
    print("\n4. Starting detailed per-pixel extraction...")
    print("   This process may take several minutes...")

    chirps_list = chirps_monthly.toList(total_images)
    timestamps = precip_df['Timestamp'].values

    pixel_data = {}
    t0 = time.time()

    for j in range(total_images):
        if j % 12 == 0:
            elapsed = time.time() - t0
            year = 1994 + j // 12
            remaining = (elapsed / (j + 1)) * (total_images - j) if j > 0 else 0
            print(
                f"   Processing year {year}: {j + 1}/{total_images} months - "
                f"Elapsed time: {elapsed:.1f}s - "
                f"Remaining time: ~{remaining / 60:.1f}min"
            )

        img = ee.Image(chirps_list.get(j))
        lat, lon, values = extract_lat_lon_pixel(img, geometry, ['precipitation'])
        pixel_data[timestamps[j]] = values[0]

    # -------------------------------------------------------------------------
    # 5. Organization and saving
    # -------------------------------------------------------------------------
    print("\n5. Organizing and saving data...")
    t0 = time.time()

    df_precip = pd.DataFrame.from_dict(pixel_data)
    df_precip = df_precip.assign(Latitude=lat, Longitude=lon)
    df_precip = df_precip.set_index(['Latitude', 'Longitude'])

    print(f"   DataFrame created: {df_precip.shape[0]} pixels x {df_precip.shape[1]} months")

    filename = f"pr_chirps_{start_date[:4]}_{end_date[:4]}.xlsx"
    df_precip.to_excel(filename, index=True)

    print(f"   Data saved to: {filename}")
    print(f"   Time to save: {time.time() - t0:.1f}s")

    # -------------------------------------------------------------------------
    # Final summary
    # -------------------------------------------------------------------------
    total_time = time.time() - t0
    print("\n=== PROCESSING COMPLETE ===")
    print(f"Total time: {total_time / 60:.1f} minutes")
    print(f"Period: {start_date} to {end_date} ({total_images} months)")
    print(f"Area: {df_precip.shape[0]} pixels")
    print(f"File generated: {filename}")

    print("\nData preview (first 5 pixels and 5 months):")
    display(df_precip.iloc[:5, :5])

    print("\nDescriptive statistics (mm/month):")
    print(f"Overall mean: {df_precip.values.mean():.2f} mm/month")
    print(f"Standard deviation: {df_precip.values.std():.2f} mm/month")
    print(f"Minimum value: {df_precip.values.min():.2f} mm/month")
    print(f"Maximum value: {df_precip.values.max():.2f} mm/month")
    print(f"Missing data: {df_precip.isnull().sum().sum()} values")


if __name__ == "__main__":
    main()

✅ GEE autenticado e inicializado!
